# Extracting weather data

In this preliminary step we need to collect weather data for Turin during the selected time period since the Weather is a very important parameter for predicting PM 2.5 value
We will use the Arpa Piemonte API to collect the necessary data.
All informations regarding the API and additional datasets can be found on the [institution's official website](https://www.arpa.piemonte.it/)

In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import requests

There are many API methods available in the [weather data API documentation](https://utility.arpa.piemonte.it/docs/#/meteoidro/meteoidro_punti_misura_meteo_list). We will be colelcting our data using the **/meteoidro/dati_giornalieri_meteo/{pk_misura}/**

the **{pk_misura}** parameter is the station's Id. Since we do not know what station to query it is necessary first to list all stations available for the targetted region and to select one. The **/meteoidro/punti_misura_meteo/** method lists all recorded stations and all sorts of information. We will be looking at the province Id since Turin's province Id is known to be 001272 and the date of first and last records to get the up to date data. The URL sent for each station can contain the Id we are looking for 

In [2]:
# used method to look for all stations
url = "https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/"

# collecting all found corresponding stations
all_stations = []

# going through all available pages returned by the API
while url:
    r = requests.get(url, headers={"accept": "application/json"})
    data = r.json()
    
    # appending to our record all stations recording data in turin
    turin = [s for s in data["results"] if s["codice_istat_comune"] == "001272"]
    all_stations.extend(turin)
    
    # when finished go to the next page
    url = data.get("next")
    print(f"Finished going through all pages, found : {len(all_stations)} corresponding weather stations")

print("\nStations Turin :")
for s in all_stations:
    print(s["denominazione"], "|", s["url"], "|", s["data_inizio_dati"], "→", s["data_fine_dati"])

Finished going through all pages, found : 7 corresponding weather stations

Stations Turin :
TORINO BUON PASTORE | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-900/ | 1989-03-22 → 2004-08-03
TORINO ITALGAS | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-901/ | 1992-11-10 → 2001-03-22
TORINO VALLERE | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-904/ | 2001-05-18 → 2025-12-31
TORINO REISS ROMOLI | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-905/ | 2003-12-18 → 2025-12-31
TORINO VIA DELLA CONSOLATA | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-906/ | 2003-12-19 → 2025-12-31
TORINO GIARDINI REALI | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-907/ | 2004-08-06 → 2025-12-31
TORINO ALENIA | https://utility.arpa.piemonte.it/meteoidro/punti_misura_meteo/PIE-001272-908/ | 2005-06-01 → 2025-12-31


Amongst these stations, the **Via Della Consolata station** is the most important one since it collects weather data close to the city center. It is also up to date so we can use this one. To access it we can use the id which is according to the url **PIE-001272-906/**. We can now move to extracting the weather data from 2022 to 2025. Unfortunately we will have to use other methods to get data for 2026 and do our testing predictions

In [ ]:
# collecting weather data
all_meteo = []

# meteo station id collecting data for weather in Turin
station_id = "PIE-001272-906"

# search parameters
start_date = "2022-01-01"
end_date = "2025-12-31" # data ends in 2025

params = {
    "fk_id_punto_misura_meteo": station_id,
    "data_min": start_date,
    "data_max": end_date,
}

# querying
url = "https://utility.arpa.piemonte.it/meteoidro/dati_giornalieri_meteo/"

# going through all returned pages of weather records
while url:
    r = requests.get(url, headers={"accept": "application/json"}, params=params)
    data = r.json()
    all_meteo.extend(data["results"])
    url = data.get("next")
    params = {} 
    print(f"{len(all_meteo)} lignes récupérées...")

df_meteo = pd.DataFrame(all_meteo)
print(df_meteo.shape)
display(df_meteo.head())

366 lignes récupérées...
732 lignes récupérées...
1098 lignes récupérées...
1461 lignes récupérées...
(1461, 27)


,url,data,tmedia,tmax,tmin,tclasse,ptot,pclasse,vmedia,vraffica,...,uclasse,rtot,rclasse,hdd_base18,hdd_base20,cdd_base18,bmedia,bmedia_slm,bclasse,fk_id_punto_misura_meteo
0,https://utility.arpa.piemonte.it/meteoidro/dat...,2022-01-01,7.9,14.2,2.9,MZ00,0.0,MZ,1.2,5.1,...,MZ00,6.5,MZ,9.5,11.5,0.0,None,None,None,https://utility.arpa.piemonte.it/meteoidro/pun...
1,https://utility.arpa.piemonte.it/meteoidro/dat...,2022-01-02,6.3,11.5,3.5,MZ00,0.0,MZ,0.8,2.7,...,MZ00,4.5,MZ,10.5,12.5,0.0,None,None,None,https://utility.arpa.piemonte.it/meteoidro/pun...
2,https://utility.arpa.piemonte.it/meteoidro/dat...,2022-01-03,6.3,9.0,3.5,MZ00,0.0,MZ,0.9,3.5,...,MZ00,3.1,MZ,11.8,13.8,0.0,None,None,None,https://utility.arpa.piemonte.it/meteoidro/pun...
3,https://utility.arpa.piemonte.it/meteoidro/dat...,2022-01-04,6.4,7.6,5.1,MZ00,0.0,MZ,0.9,2.5,...,MZ00,0.6,MY,11.7,13.7,0.0,None,None,None,https://utility.arpa.piemonte.it/meteoidro/pun...
4,https://utility.arpa.piemonte.it/meteoidro/dat...,2022-01-05,7.5,10.4,5.8,MZ00,0.4,MY,1.0,5.4,...,MZ00,4.6,MZ,9.9,11.9,0.0,None,None,None,https://utility.arpa.piemonte.it/meteoidro/pun...


Taking now a closer look at the columns to only take the interesting data

In [6]:
print(df_meteo.columns)

Index(['url', 'data', 'tmedia', 'tmax', 'tmin', 'tclasse', 'ptot', 'pclasse',
       'vmedia', 'vraffica', 'settore_prevalente', 'tempo_permanenza',
       'durata_calma', 'vclasse', 'umedia', 'umin', 'umax', 'uclasse', 'rtot',
       'rclasse', 'hdd_base18', 'hdd_base20', 'cdd_base18', 'bmedia',
       'bmedia_slm', 'bclasse', 'fk_id_punto_misura_meteo'],
      dtype='str')


We will only be taking the features which may help us predict the value of PM2.5, in other words weather data related to temperature,rain, wind, humidity, sun intenisty; While the air pressure would have been nice to have, the records only show NaN values

In [11]:
interesting_columns = features_meteo = [
    # date
    "data",
    # temperatures
    "tmedia", "tmax", "tmin",
    # rain
    "ptot",
    # wind
    "vmedia", "vraffica", "settore_prevalente",
    # humidity
    "umedia",
    # sun intensity
    "rtot",
    
]

In [12]:
df_meteo = df_meteo[interesting_columns]
df_meteo.head()

,data,tmedia,tmax,tmin,ptot,vmedia,vraffica,settore_prevalente,umedia,rtot
0,2022-01-01,7.9,14.2,2.9,0.0,1.2,5.1,SW,81.0,6.5
1,2022-01-02,6.3,11.5,3.5,0.0,0.8,2.7,SW,84.0,4.5
2,2022-01-03,6.3,9.0,3.5,0.0,0.9,3.5,NNE,84.0,3.1
3,2022-01-04,6.4,7.6,5.1,0.0,0.9,2.5,NNE,98.0,0.6
4,2022-01-05,7.5,10.4,5.8,0.4,1.0,5.4,SSW,85.0,4.6


In [ ]:
# saving everything to avoid to fasten process on next execution
filename = "meteo.csv"
df_meteo.to_csv(filename, index= False)

In order to complete the dataset with the data of 2026, we can just use Open Meteo API.

In [ ]:
# 1. Convertir settore_prevalente en degrés dans df_meteo
direction_map = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5,
    "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5,
}

df_meteo = pd.read_csv("meteo.csv")
df_meteo["wind_deg"] = df_meteo["settore_prevalente"].map(direction_map)
df_meteo = df_meteo.drop("settore_prevalente", axis=1)

# 2. Appel Open-Meteo pour jan-mars 2026 uniquement
r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 45.0703,
        "longitude": 7.6869,
        "start_date": "2026-01-01",
        "end_date": "2026-03-23",
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "wind_speed_10m_mean",
            "wind_speed_10m_max",
            "wind_direction_10m_dominant",
            "relative_humidity_2m_mean",
            "shortwave_radiation_sum",
        ],
        "timezone": "Europe/Rome"
    }
)

df_2026 = pd.DataFrame(r.json()["daily"])

# 3. Renommer pour matcher exactement les colonnes de df_meteo
df_2026 = df_2026.rename(columns={
    "time":                        "data",
    "temperature_2m_mean":         "tmedia",
    "temperature_2m_max":          "tmax",
    "temperature_2m_min":          "tmin",
    "precipitation_sum":           "ptot",
    "wind_speed_10m_mean":         "vmedia",
    "wind_speed_10m_max":          "vraffica",
    "wind_direction_10m_dominant": "wind_deg",
    "relative_humidity_2m_mean":   "umedia",
    "shortwave_radiation_sum":     "rtot",
})

# 4. Concaténer et trier
df_final = pd.concat([df_meteo, df_2026], ignore_index=True)
df_final = df_final.sort_values("data").reset_index(drop=True)

print(df_final.shape)
print(f"Période : {df_final['data'].min()} → {df_final['data'].max()}")
display(df_final.tail())

In [4]:
df_final.isna().sum()

date               0
temp_mean          0
temp_max           0
temp_min           0
precipitation      0
wind_speed_mean    0
wind_speed_max     0
humidity_mean      0
solar_radiation    0
wind_sin           0
wind_cos           0
dtype: int64

In [11]:
df_final = df_final.rename(columns={
    "data":     "date",
    "tmedia":   "temp_mean",
    "tmax":     "temp_max",
    "tmin":     "temp_min",
    "ptot":     "precipitation",
    "vmedia":   "wind_speed_mean",
    "vraffica": "wind_speed_max",
    "umedia":   "humidity_mean",
    "rtot":     "solar_radiation",
    "wind_sin": "wind_sin",
    "wind_cos": "wind_cos",
})

print(df_final.columns.tolist())

['date', 'temp_mean', 'temp_max', 'temp_min', 'precipitation', 'wind_speed_mean', 'wind_speed_max', 'humidity_mean', 'solar_radiation', 'wind_sin', 'wind_cos']


In [6]:
# saving everything to avoid to fasten process on next execution
filename = "meteo.csv"
df_final.to_csv(filename, index= False)